# Krippendorff's Alpha: Agreement That Generalises

**Computational Text Analysis Pack · Module 17 — Annotation and Agreement**

*Technique:* Alpha for any number of coders, missing data and measurement levels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/17b_krippendorffs_alpha.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Cohen's kappa handles exactly two coders, who must both code everything, on
nominal categories. Real annotation projects break all three assumptions: three
readers, each covering an overlapping subset, sometimes on an ordered scale.

**Krippendorff's alpha** is the general answer. One coefficient, any number of
coders, missing data allowed, and a choice of distance function for nominal,
ordinal or interval data. It is the statistic content analysis actually reports.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# Three coders with different reliability, and genuine gaps: each skips some
# documents, exactly as happens when annotation is split across a team.
rng = np.random.default_rng(9)
sample = df.sample(n=300, random_state=2).reset_index(drop=True)
gold = sample["category"].tolist()
cats = sorted(set(gold))

def make_coder(accuracy, skip, seed):
    r = np.random.default_rng(seed)
    out = []
    for label in gold:
        if r.random() < skip:
            out.append(None)
        elif r.random() < accuracy:
            out.append(label)
        else:
            out.append(r.choice(cats))
    return out

coders = {"A": make_coder(0.85, 0.05, 1),
          "B": make_coder(0.75, 0.20, 2),
          "C": make_coder(0.60, 0.35, 3)}
for name, vals in coders.items():
    print(f"coder {name}: coded {sum(v is not None for v in vals):>3}/{len(gold)}")

## Computing alpha

Alpha compares **observed disagreement** against the disagreement you would
expect if the labels were shuffled across the whole pool. The units are
disagreement rather than agreement, which is why it handles missing data
gracefully: a document coded by only one person contributes nothing to observed
disagreement and is simply skipped.

In [ ]:
def krippendorff_alpha(coder_values, level="nominal"):
    """coder_values: dict of coder -> list of labels (None where not coded)."""
    n_units = len(next(iter(coder_values.values())))
    units = []
    for i in range(n_units):
        vals = [c[i] for c in coder_values.values() if c[i] is not None]
        if len(vals) > 1:
            units.append(vals)

    def delta(a, b):
        if level == "nominal":
            return 0.0 if a == b else 1.0
        return (float(a) - float(b)) ** 2

    # Observed disagreement.
    Do_num = 0.0
    total_pairable = 0
    for vals in units:
        m = len(vals)
        total_pairable += m
        for i in range(m):
            for j in range(m):
                if i != j:
                    Do_num += delta(vals[i], vals[j]) / (m - 1)
    Do = Do_num / total_pairable

    # Expected disagreement, over the whole pool of assigned values.
    pool = [v for vals in units for v in vals]
    N = len(pool)
    De_num = 0.0
    counts = Counter(pool)
    for a, ca in counts.items():
        for b, cb in counts.items():
            if a == b:
                De_num += ca * (cb - 1) * delta(a, b)
            else:
                De_num += ca * cb * delta(a, b)
    De = De_num / (N * (N - 1))
    return 1 - Do / De if De else 1.0

alpha = krippendorff_alpha(coders)
print(f"Krippendorff's alpha, all three coders : {alpha:.3f}")
print()
print("Krippendorff's own guidance: rely on data above 0.800, and treat 0.667")
print("as the lowest value on which to draw tentative conclusions.")

In [ ]:
# Which coder is dragging the number down?
print(f"{'coders included':<26}{'alpha':>8}")
print("-" * 34)
print(f"{'A + B + C':<26}{alpha:>8.3f}")
for drop in coders:
    subset = {k: v for k, v in coders.items() if k != drop}
    print(f"{'without ' + drop:<26}{krippendorff_alpha(subset):>8.3f}")
print()
print("If alpha rises sharply when one coder is removed, that coder needs")
print("retraining or the manual needs a clearer rule — this is a diagnostic,")
print("not a licence to delete the inconvenient annotator.")

## Why the measurement level matters

Alpha's distance function encodes what kind of scale you are on. On an ordinal
scale — say a five-point judgement of how negatively a passage portrays its
subject — coding 4 when someone else coded 5 is a small disagreement, and coding
1 is a large one. Nominal alpha treats both as simply "different", and throws
that information away.

In [ ]:
r = np.random.default_rng(21)
truth = r.integers(1, 6, size=250)
ord_coders = {}
for name, noise in [("A", 0.5), ("B", 0.9), ("C", 1.6)]:
    vals = np.clip(np.round(truth + r.normal(0, noise, size=len(truth))), 1, 5)
    ord_coders[name] = [int(v) for v in vals]

print(f"nominal alpha (near-misses count as full disagreement) : "
      f"{krippendorff_alpha(ord_coders, 'nominal'):.3f}")
print(f"interval alpha (near-misses count as small)            : "
      f"{krippendorff_alpha(ord_coders, 'interval'):.3f}")
print()
print("Same data, same coders. Reporting nominal alpha on an ordered scale")
print("understates reliability, sometimes badly enough to sink a study that was")
print("actually fine.")

### 🔧 Your turn 1

Raise coder C's skip rate to 0.7 so they code very little. What happens to alpha, and would kappa have coped at all?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Alpha barely moves, because C now contributes to few pairable units and is
weighted accordingly — handling exactly this case is why alpha exists. Cohen's
kappa could not be computed at all across three coders with missing data; you
would be forced into pairwise kappas on the overlapping subsets and then into an
indefensible average of them. That is the practical argument for alpha, and the
reason it is the coefficient content analysis settled on.

</details>